# Skin cancer medical chat assistant — Anthropic (Claude) experiment notebook

This notebook mirrors `medical_chat_assistant_demo.ipynb` (OpenAI) and `medical_chat_assistant_gemini_demo.ipynb` (Gemini): same **skin-lesion triage** framing, **patient profile**, **multi-turn chat**, **optional vision**, and **JSON transcript** export — using the **Anthropic Messages API** (`anthropic` SDK).

**Objectives covered**
- Supportive chat for **skin lesion concerns** and **triage-style decision support** (not a replacement for a clinician)
- **Tracking / monitoring** style prompts
- **Local transcript** with timestamps and export
- **Continuation** via client-side `messages` history passed to each `messages.create` call
- **Personalized mentor** tone via `system` + patient profile
- **Image review** (base64 image blocks; URL fetch for demos)

---

**Setup:** Add **`ANTHROPIC_API_KEY`** to `.env` at the project root. Optionally set **`ANTHROPIC_MEDICAL_MODEL`** (see env cell). Do not commit API keys.

**Important:** This is a **demonstration**. The model is **not** a licensed medical professional. It must **not** be used as a sole source for diagnosis or treatment. Skin cancer evaluation often requires an in-person exam and sometimes dermoscopy/biopsy. Encourage users to consult qualified healthcare providers and emergency services when appropriate.

## 1. Dependencies

Run once if needed:

`pip install anthropic python-dotenv`

In [1]:
from __future__ import annotations

import base64
import json
import os
import urllib.error
import urllib.request
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import anthropic
from dotenv import load_dotenv

## 2. Environment and model

Loads `.env` from the project root (`notebooks/` parent if that is your cwd). Override the model with **`ANTHROPIC_MEDICAL_MODEL`** (must support vision if you use `chat_with_image`). **`ANTHROPIC_MAX_TOKENS`** caps each assistant reply (default 2048).

In [2]:
_here = Path.cwd().resolve()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY is not set. Add it to .env or the environment.")

MODEL = os.getenv("ANTHROPIC_MEDICAL_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = int(os.getenv("ANTHROPIC_MAX_TOKENS", "2048"))

client = anthropic.Anthropic()

## 3. System instructions and patient profile

The combined text is sent as the **`system`** parameter on every request. If you change the profile mid-notebook, call **`reset_conversation()`** so behavior stays consistent (history was conditioned on the old system text).

In [3]:
MEDICAL_ASSISTANT_INSTRUCTIONS = """You are a supportive health-information assistant focused on **skin cancer / suspicious skin lesions**.

Scope (skin):
- Help users discuss skin spots/moles/rashes/lesions in a structured way.
- Provide **triage-style guidance** (what to monitor, what to ask a clinician, when to seek urgent evaluation).
- Support longitudinal tracking (size, color, symptoms) and healthy sun-safety habits.

Safety and boundaries:
- You are NOT a doctor. Do NOT provide a definitive diagnosis (e.g., do not say "this is melanoma").
- Make it clear that skin cancer assessment often requires an in-person exam and sometimes dermoscopy/biopsy.
- Use cautious language ("could be", "possible", "worth getting checked").
- If there are urgent red flags (rapidly changing lesion, bleeding/ulceration, severe pain, signs of infection, systemic symptoms, immunosuppression + concerning lesion), advise prompt in-person care.
- If the user reports severe symptoms or is otherwise unwell, advise urgent/emergency care as appropriate.

Image handling (when an image is provided):
- Describe only what is visible. Do not claim certainty from a photo.
- Use evidence-based framing like **ABCDE** (Asymmetry, Border, Color, Diameter, Evolving) and the "ugly duckling" sign.
- Ask for key context: duration, changes, symptoms (itching, pain, bleeding), location, personal/family history, sun exposure, and any prior skin cancers.

Style:
- Be empathetic, concise, and clear. Ask clarifying questions when it improves safety and usefulness.
- When recommending next steps, be specific (e.g., "book a dermatology visit", "take standardized photos weekly", "measure with a ruler").
- The response should be very short and concise for the user to understand the next steps. Use 50–75 max words per one message.
"""


@dataclass
class PatientProfile:
    display_name: str = "Alex"
    age: Optional[int] = None
    chronic_conditions: str = ""
    monitoring_goals: str = ""
    communication_style: str = "warm, encouraging mentor"
    extra_notes: str = ""

    def personalization_block(self) -> str:
        parts = [
            f"Address the user as {self.display_name}.",
            f"Communication style: {self.communication_style}.",
        ]
        if self.age is not None:
            parts.append(f"Patient-stated age: {self.age}.")
        if self.chronic_conditions.strip():
            parts.append(f"Patient-stated conditions / history: {self.chronic_conditions.strip()}")
        if self.monitoring_goals.strip():
            parts.append(f"Current monitoring / goals: {self.monitoring_goals.strip()}")
        if self.extra_notes.strip():
            parts.append(f"Additional context: {self.extra_notes.strip()}")
        return "\n".join(parts)


def build_system_prompt(profile: PatientProfile) -> str:
    return (
        MEDICAL_ASSISTANT_INSTRUCTIONS
        + "\n\n---\nPersonalization:\n"
        + profile.personalization_block()
    )

## 4. Chat session: history, continuation, export

- **Continuation:** `self._messages` holds alternating **user** / **assistant** turns in Anthropic API shape; each call sends **full** history plus `system`.
- **New topic:** `reset_conversation()` clears history (profile unchanged unless you edit it).
- **Vision:** images are sent as **base64** blocks (local path or downloaded URL).

In [4]:
def _guess_image_mime(path: Path) -> str:
    ext = path.suffix.lower()
    return {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }.get(ext, "image/jpeg")


def fetch_image_url(url: str, timeout_s: float = 60.0) -> tuple[bytes, str]:
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "MedicalChatAnthropicDemo/1.0"},
        method="GET",
    )
    with urllib.request.urlopen(req, timeout=timeout_s) as resp:
        data = resp.read()
        raw_ct = resp.headers.get_content_type() or "application/octet-stream"
    mime = raw_ct.split(";")[0].strip().lower()
    if mime == "application/octet-stream":
        mime = "image/jpeg"
    return data, mime


def _message_text(msg: Any) -> str:
    parts: list[str] = []
    for block in msg.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    return "".join(parts).strip()


def _assistant_content_for_history(msg: Any) -> list[dict[str, Any]]:
    """Replayable assistant blocks for the next Messages API call (text only for this demo)."""
    out: list[dict[str, Any]] = []
    for block in msg.content:
        if getattr(block, "type", None) == "text":
            out.append({"type": "text", "text": block.text})
    if not out:
        out.append({"type": "text", "text": "(no text content)"})
    return out


class MedicalAnthropicChatSession:
    """Skin-lesion demo chat using Anthropic Messages API with client-side history."""

    def __init__(
        self,
        anthropic_client: anthropic.Anthropic,
        profile: PatientProfile,
        *,
        model: str = MODEL,
        max_tokens: int = MAX_TOKENS,
    ) -> None:
        self.client = anthropic_client
        self.profile = profile
        self.model = model
        self.max_tokens = max_tokens
        self.transcript: list[dict[str, Any]] = []
        self._messages: list[dict[str, Any]] = []

    @property
    def system_prompt(self) -> str:
        return build_system_prompt(self.profile)

    def reset_conversation(self) -> None:
        self._messages = []

    def _append_transcript(self, role: str, content: str, meta: Optional[dict[str, Any]] = None) -> None:
        entry: dict[str, Any] = {
            "role": role,
            "content": content,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
        if meta:
            entry["meta"] = meta
        self.transcript.append(entry)

    def _complete(self, user_content: str | list[dict[str, Any]]) -> str:
        self._messages.append({"role": "user", "content": user_content})
        resp = self.client.messages.create(
            model=self.model,
            max_tokens=self.max_tokens,
            system=self.system_prompt,
            messages=self._messages,
        )
        self._messages.append({"role": "assistant", "content": _assistant_content_for_history(resp)})
        text = _message_text(resp)
        return text

    def chat(self, user_text: str) -> str:
        text = self._complete(user_text)
        self._append_transcript("user", user_text)
        self._append_transcript("assistant", text)
        return text

    def chat_with_image(
        self,
        user_text: str,
        *,
        image_path: Optional[str | Path] = None,
        image_url: Optional[str] = None,
    ) -> str:
        if image_path is not None:
            p = Path(image_path)
            raw = p.read_bytes()
            mime = _guess_image_mime(p)
        elif image_url is not None:
            try:
                raw, mime = fetch_image_url(image_url)
            except urllib.error.URLError as e:
                raise RuntimeError(f"Failed to download image URL: {e}") from e
        else:
            raise ValueError("Provide image_path or image_url")

        b64 = base64.standard_b64encode(raw).decode("ascii")
        user_content: list[dict[str, Any]] = [
            {
                "type": "image",
                "source": {"type": "base64", "media_type": mime, "data": b64},
            },
            {"type": "text", "text": user_text},
        ]
        text = self._complete(user_content)
        self._append_transcript("user", user_text, meta={"has_image": True})
        self._append_transcript("assistant", text)
        return text

    def save_transcript(self, path: str | Path) -> Path:
        path = Path(path)
        path.write_text(
            json.dumps(
                {
                    "provider": "anthropic",
                    "model": self.model,
                    "profile": self.profile.__dict__,
                    "transcript": self.transcript,
                },
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        return path

    def load_transcript(self, path: str | Path) -> None:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
        self.transcript = data.get("transcript", [])
        # API message history is not restored; use reset + new chat for a clean thread.

## 5. Instantiate profile and session

In [5]:
profile = PatientProfile(
    display_name="Alex",
    age=42,
    chronic_conditions="Seasonal allergies (patient-reported).",
    monitoring_goals="Daily energy and sleep check-in for two weeks.",
    communication_style="warm mentor; short paragraphs",
)

session = MedicalAnthropicChatSession(client, profile)
print("Model:", MODEL)
print("System prompt preview:\n", session.system_prompt[:400], "...")

Model: claude-haiku-4-5-20251001
System prompt preview:
 You are a supportive health-information assistant focused on **skin cancer / suspicious skin lesions**.

Scope (skin):
- Help users discuss skin spots/moles/rashes/lesions in a structured way.
- Provide **triage-style guidance** (what to monitor, what to ask a clinician, when to seek urgent evaluation).
- Support longitudinal tracking (size, color, symptoms) and healthy sun-safety habits.

Safety  ...


## 6. Demo: skin lesion check-in (monitoring + triage)

In [6]:
import time

t0 = time.time()
reply = session.chat(
    "Skin check-in: I noticed a mole on my upper arm that seems a bit darker than the others. "
    "I'm not sure if it has changed, but it's about the size of a pencil eraser. It doesn't hurt. "
    "What questions should you ask me to assess risk, and what should I monitor over the next 2 weeks?"
)
t1 = time.time()
print(reply)
print(f"Inference time: {t1 - t0:.2f}s")

Hey Alex, great that you're paying attention! Let me gather some key details:

**Quick questions:**
1. How long have you noticed this mole? (weeks, months, years?)
2. Has it changed in size, color, or shape recently—or is this your first time noticing it?
3. Are the edges sharp or blurry/irregular?
4. Is the color uniform or mixed (browns, blacks, reds)?
5. Any itching, bleeding, or tenderness?
6. History of sunburns or tanning bed use on your arms?

**What to monitor next 2 weeks:**
- Take a **dated photo** with a ruler beside it (baseline).
- Check weekly for size/color shifts or new symptoms.
- Avoid sun exposure to that area; use SPF 30+.

**If any of these appear: seek a dermatology visit promptly:**
- Rapid growth, bleeding, or increasing itch.
- Borders becoming irregular.
- New colors emerging.

Otherwise, a routine skin check with your doctor in the next month is sensible. Sound reasonable?
Inference time: 3.59s


## 7. Demo: continuation (same client-side history)

In [7]:
follow_up = session.chat(
    "Update: I compared it to my other moles and this one looks a bit more irregular at the edges. "
    "It also seems like it might have gotten slightly darker over the last month (not sure). "
    "What specific red flags should push me to book a dermatology visit sooner rather than later?"
)
print(follow_up)

Good observation, Alex. **Irregular edges + possible darkening over a month** warrants closer attention.

**Book dermatology *soon* (within 1–2 weeks) if:**
- Edges become noticeably jagged or asymmetrical.
- Colors mixing (brown, black, red, tan in one lesion).
- Size growing visibly (compare your photo to last week's).
- Itching, bleeding, or oozing starts.
- Diameter exceeds a pencil eraser (about 6mm).

**Seek *urgent* care (same day) if:**
- Rapid growth, active bleeding, or open sore.
- Severe pain or signs of infection.

**Right now:** You don't describe urgent symptoms, but the combo of **irregularity + darkening trend** is worth a dermatology eval sooner rather than waiting. This could be benign, but a pro's eyes (possibly with dermoscopy) beat home monitoring alone.

Take a ruler photo *today*, then book that appointment. How does that feel?


## 8. Demo: image-assisted skin lesion discussion (vision)

Replace with a **local** lesion photo path. The same `org_2_5.jpg` convention as the other demos is used when the file exists under the project root.

Privacy: skin photos can be identifying; use de-identified images and appropriate consent.

In [8]:
DEMO_IMAGE_PATH = PROJECT_ROOT / "org_2_5.jpg"

if DEMO_IMAGE_PATH.is_file():
    vision_reply = session.chat_with_image(
        "Please look at this skin photo and do a careful, non-diagnostic review using the ABCDE framework. "
        "List what would make this more vs less concerning, what extra context you need, and what the safest next steps are. "
        "Do NOT provide a definitive diagnosis from the image.",
        image_path=DEMO_IMAGE_PATH,
    )
    print(vision_reply)
else:
    print("Skip: not found —", DEMO_IMAGE_PATH, "(add a local JPG/PNG under the project root or set image_path).")

**ABCDE review of your image:**

**Asymmetry:** Slightly irregular shape—one side fuller than the other. ⚠️

**Border:** Edges appear somewhat scalloped/uneven rather than sharp. ⚠️

**Color:** Uniform reddish-brown; no mixed colors (black, tan patches). ✓

**Diameter:** Appears ~6–7mm (pencil eraser range). Borderline.

**Evolving:** You reported darkening over 1 month—key concern.

---

**What raises concern:**
- Asymmetry + irregular border + reported darkening trend.

**What's reassuring:**
- Single, uniform color (no variegation).
- No bleeding/ulceration visible.
- No severe symptoms reported.

**Context I still need:**
- Sun exposure history on upper arm?
- Any family history of skin cancer?
- How confident are you in the "darkening" (clear comparison photo)?

---

**Safest next step:**
**Book a dermatology visit *this week or next.*** A clinician can examine it in person, possibly use dermoscopy, and clarify whether monitoring or biopsy is needed.

A photo alone can't rule out 

## 9. Export transcript to JSON

In [9]:
out_path = PROJECT_ROOT / "notebooks" / "medical_chat_transcript_anthropic.json"
saved = session.save_transcript(out_path)
print("Saved:", saved)

Saved: /home/nipuna/medical_chat_bot_dev/notebooks/medical_chat_transcript_anthropic.json


## 10. Optional: new conversation thread

In [10]:
session.reset_conversation()
print(session.chat("Let's start a new topic: how do I prepare questions for my annual physical?"))

Hi Alex! I appreciate the question, but **annual physicals aren't my specialty**—I'm focused on skin health and suspicious lesions.

That said, if you have any **skin concerns** (moles, rashes, spots) you want to raise during that physical, I'm happy to help you prepare those questions.

For general physical prep, a primary care or wellness resource would serve you better.

**Any skin concerns on your mind?**
